In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!apt-get install -y r-base r-base-dev libcurl4-openssl-dev libssl-dev libxml2-dev > /dev/null 2>&1
!pip install -q rpy2
%load_ext rpy2.ipython

In [ ]:
# ==================== ANTHRAX BOOTSTRAP — SINGLE CELL ====================
from google.colab import drive; drive.mount('/content/drive')
import subprocess
subprocess.run("apt-get install -y r-base r-base-dev libcurl4-openssl-dev libssl-dev libxml2-dev",
               shell=True, capture_output=True)
subprocess.run("pip install -q rpy2", shell=True)
import rpy2.robjects as ro

ro.r('''
for (p in c("pvclust","data.table","ggplot2","png","gridExtra"))
  if (!requireNamespace(p, quietly=TRUE)) install.packages(p, repos="https://cloud.r-project.org")
suppressPackageStartupMessages({library(data.table);library(pvclust);library(ggplot2)
  library(grid);library(gridExtra);library(png)})

ROOT <- "/content/drive/MyDrive/research/Anthrax/Anthraxpaper/"
EV   <- paste0(ROOT,"External_Validation/")
RES  <- paste0(ROOT,"Final/Bootstrap_Final/")
FIG  <- paste0(RES,"Figures/"); TAB <- paste0(RES,"Tables/"); RDA <- paste0(RES,"RData/")
for (d in c(RES,FIG,TAB,RDA)) dir.create(d, recursive=TRUE, showWarnings=FALSE)

LAB <- c(GSM359332=0,GSM359753=0,GSM359754=0,GSM359755=0,GSM359758=1,GSM359759=1,GSM359760=1,
         GSM305784=0,GSM305785=0,GSM305786=1,GSM305787=1)
COMMON <- c("GULP1","GATAD2A","ACSS1","LOC100506530","CNTN4","LINC00485",
            "WNT6","LOC101928461","IL20RA","VWC2","RNF128","TTC6")
HUB  <- c("WNT6","FZD9","DKK4","FZD6","FZD3")
FEAT <- unique(c(COMMON,HUB)); NBOOT <- 1000; NCLOUD <- 500; set.seed(1)

## filtered_data_*.csv থেকে probe -> gene collapse (GSM নাম ঠিক থাকে)
read_filtered <- function(path){
  df <- as.data.frame(fread(path))
  sym <- grep("^Gene[._ ]?symbol$|^Symbol$", colnames(df), ignore.case=TRUE, value=TRUE)[1]
  gsm <- grep("^GSM[0-9]+", colnames(df), value=TRUE)
  stopifnot(!is.na(sym), length(gsm) > 0)
  d <- df[, c(sym, gsm)]
  s <- trimws(as.character(d[[sym]]))
  keep <- s != "" & !is.na(s) & s != "NA" & !grepl("///", s, fixed=TRUE)
  d <- d[keep,]; s <- s[keep]
  d[gsm] <- lapply(d[gsm], as.numeric)
  m <- rowsum(as.matrix(d[gsm]), s) / as.vector(table(s)[rownames(rowsum(as.matrix(d[gsm]), s))])
  as.data.frame(m)
}
A <- read_filtered(paste0(EV,"Dis_GSE12131/filtered_data_GSE12131.csv"))
B <- read_filtered(paste0(EV,"Dis_GSE14390/filtered_data_GSE14390.csv"))
cat("GSE12131:",nrow(A),"genes x",ncol(A),"samples\\n")
cat("GSE14390:",nrow(B),"genes x",ncol(B),"samples\\n")

if (max(A,na.rm=TRUE) > 50) { A <- log2(A+1); cat("GSE12131: log2 applied\\n") }
if (max(B,na.rm=TRUE) > 50) { B <- log2(B+1); cat("GSE14390: log2 applied\\n") }

feat <- intersect(FEAT, intersect(rownames(A), rownames(B)))
cat("features available:",length(feat),"of",length(FEAT),"\\n")
missing <- setdiff(FEAT, feat); if (length(missing)) cat("missing:",paste(missing,collapse=", "),"\\n")

M <- cbind(A[feat,,drop=FALSE], B[feat,,drop=FALSE])
Z <- t(scale(t(M))); Z[is.na(Z)] <- 0
origin <- c(rep("GSE12131",ncol(A)),rep("GSE14390",ncol(B))); names(origin) <- colnames(Z)
group  <- ifelse(LAB[colnames(Z)]==1,"Anthrax","Control")
cat("samples:",ncol(Z)," anthrax:",sum(group=="Anthrax")," control:",sum(group=="Control"),"\\n")

## PANEL A — real gene-resampling bootstrap
cloud <- do.call(rbind, lapply(seq_len(NCLOUD), function(b){
  idx <- sample(nrow(Z), nrow(Z), replace=TRUE)
  mds <- cmdscale(dist(t(Z[idx,,drop=FALSE])), k=2)
  data.frame(Coord1=mds[,1],Coord2=mds[,2],Sample=rownames(mds),
             Dataset=origin[rownames(mds)],Group=group[rownames(mds)]) }))
pA <- ggplot(cloud,aes(Coord1,Coord2,colour=Group,shape=Dataset))+
  geom_point(size=1.1,alpha=.25)+
  stat_ellipse(aes(group=Sample),type="norm",linewidth=.3,colour="grey40")+
  scale_colour_manual(values=c(Control="#2E86C1",Anthrax="#C0392B"))+
  theme_classic(base_size=12)+
  labs(x="Coordinate 1",y="Coordinate 2",
       subtitle=paste(NCLOUD,"gene-resampling bootstrap replicates per sample"))
ggsave(paste0(FIG,"Bootstrap_PanelA_MDS.png"),pA,width=6.5,height=5.5,dpi=600,bg="white")

## PANEL B — multiscale bootstrap dendrogram
pv <- pvclust(Z, method.hclust="average", method.dist="correlation", nboot=NBOOT, parallel=FALSE)
png(paste0(FIG,"Bootstrap_PanelB_dendrogram.png"),width=9,height=6,units="in",res=600)
par(mar=c(9,4,3,2)); plot(pv,main="Bootstrap-resampled sample dendrogram (AU/BP %)",cex=.8,cex.pv=.7)
pvrect(pv,alpha=0.95); dev.off()
e <- pv$edges
boot <- data.frame(Cluster_edge=seq_len(nrow(e)),AU=round(e$au,4),BP=round(e$bp,4),
  SE_AU=round(e$se.au,4),P_value=signif(1-e$au,3),
  Supported_AU95=ifelse(e$au>=0.95,"Yes","No"))
write.csv(boot, paste0(TAB,"Table_bootstrap_confidence.csv"), row.names=FALSE); print(boot)

## hub-only
ph <- intersect(HUB, rownames(Z))
if (length(ph) >= 3){
  pvh <- pvclust(Z[ph,,drop=FALSE],method.hclust="average",method.dist="correlation",
                 nboot=NBOOT,parallel=FALSE)
  png(paste0(FIG,"Bootstrap_hub_only_dendrogram.png"),width=9,height=6,units="in",res=600)
  par(mar=c(9,4,3,2)); plot(pvh,main="Bootstrap dendrogram — top-5 hub genes",cex=.8,cex.pv=.7)
  pvrect(pvh,alpha=0.95); dev.off()
  write.csv(data.frame(Cluster_edge=seq_len(nrow(pvh$edges)),AU=round(pvh$edges$au,4),
            BP=round(pvh$edges$bp,4)), paste0(TAB,"Table_bootstrap_hub_only.csv"), row.names=FALSE) }

## combined
imgA <- rasterGrob(readPNG(paste0(FIG,"Bootstrap_PanelA_MDS.png")),interpolate=TRUE)
imgB <- rasterGrob(readPNG(paste0(FIG,"Bootstrap_PanelB_dendrogram.png")),interpolate=TRUE)
lb <- function(g,t) arrangeGrob(g, top=textGrob(t,x=unit(.02,"npc"),hjust=0,
                                gp=gpar(fontsize=20,fontface="bold")))
png(paste0(FIG,"Fig_Bootstrap_combined.png"),width=16,height=7,units="in",res=400)
grid.arrange(lb(imgA,"A"),lb(imgB,"B"),ncol=2,widths=c(1,1.1)); dev.off()
pdf(paste0(FIG,"Fig_Bootstrap_combined.pdf"),width=16,height=7)
grid.arrange(lb(imgA,"A"),lb(imgB,"B"),ncol=2,widths=c(1,1.1)); dev.off()

save(pv,boot,Z,origin,group,file=paste0(RDA,"Bootstrap_results.RData"))
cat("\\nBootstrap complete -> ",RES,"\\n")
''')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


(as ‘lib’ is unspecified)







	‘/tmp/Rtmppub1Jx/downloaded_packages’

(as ‘lib’ is unspecified)







	‘/tmp/Rtmppub1Jx/downloaded_packages’



GSE12131: 20848 genes x 4 samples
GSE14390: 20848 genes x 7 samples
GSE12131: log2 applied
features available: 16 of 16 
samples: 11  anthrax: 5  control: 6 
Bootstrap (r = 0.5)... Done.
Bootstrap (r = 0.56)... Done.
Bootstrap (r = 0.69)... Done.
Bootstrap (r = 0.75)... Done.
Bootstrap (r = 0.88)... Done.
Bootstrap (r = 1.0)... Done.
Bootstrap (r = 1.06)... Done.
Bootstrap (r = 1.19)... Done.
Bootstrap (r = 1.25)... Done.
Bootstrap (r = 1.38)... Done.
   Cluster_edge     AU     BP  SE_AU P_value Supported_AU95
1             1 0.7233 0.6682 0.0262  0.2770             No
2             2 0.9588 0.7897 0.0074  0.0412            Yes
3             3 0.9672 0.5669 0.0057  0.0328            Yes
4             4 0.9254 0.6959 0.0112  0.0746             No
5             5 0.8454 0.5429 0.0183  0.1550             No
6             6 0.7715 0.4734 0.0233  0.2290             No
7             7 0.7603 0.4698 0.0239  0.2400             No
8             8 0.9211 0.5786 0.0114  0.0789             No
9   

#final

In [ ]:
# ============================ ANTHRAX BOOTSTRAP — SINGLE CELL ============================
from google.colab import drive; drive.mount('/content/drive')
import subprocess; subprocess.run("pip install -q rpy2", shell=True)
import rpy2.robjects as ro

ro.r('''
for (p in c("pvclust","data.table","ggplot2","png","gridExtra"))
  if (!requireNamespace(p, quietly=TRUE)) install.packages(p, repos="https://cloud.r-project.org")
suppressPackageStartupMessages({library(data.table);library(pvclust);library(ggplot2)
  library(grid);library(gridExtra);library(png)})

ROOT  <- "/content/drive/MyDrive/research/Anthrax/Anthraxpaper/"
DATA  <- paste0(ROOT,"Datasets/"); FINAL <- paste0(ROOT,"Final/")
FIG <- paste0(FINAL,"Figures/"); TAB <- paste0(FINAL,"Tables/")
SUP <- paste0(FINAL,"Supplementary/"); RDA <- paste0(FINAL,"RData/")
for (d in c(FINAL,FIG,TAB,SUP,RDA)) dir.create(d, recursive=TRUE, showWarnings=FALSE)

LAB <- c(GSM359332=0,GSM359753=0,GSM359754=0,GSM359755=0,GSM359758=1,GSM359759=1,GSM359760=1,
         GSM305784=0,GSM305785=0,GSM305786=1,GSM305787=1)
COMMON <- c("GULP1","GATAD2A","ACSS1","LOC100506530","CNTN4","LINC00485",
            "WNT6","LOC101928461","IL20RA","VWC2","RNF128","TTC6")
HUB <- c("WNT6","FZD9","DKK4","FZD6","FZD3"); FEAT <- unique(c(COMMON,HUB))
NBOOT <- 1000; NCLOUD <- 500; set.seed(1)

read_expr <- function(p){ dt <- fread(p); rn <- dt[[1]]; dt[[1]] <- NULL
  m <- as.data.frame(dt); rownames(m) <- make.unique(as.character(rn)); m[] <- lapply(m,as.numeric); m }
A <- read_expr(paste0(DATA,"GSE12131_gene_expression.csv"))
B <- read_expr(paste0(DATA,"GSE14390_gene_expression.csv"))
feat <- intersect(FEAT, intersect(rownames(A),rownames(B)))
cat("features:",length(feat),"of",length(FEAT),"\\n")
M <- cbind(A[feat,,drop=FALSE], B[feat,,drop=FALSE])
if (max(M,na.rm=TRUE)>50) M <- log2(M+1)
Z <- t(scale(t(M))); Z[is.na(Z)] <- 0
origin <- c(rep("GSE12131",ncol(A)),rep("GSE14390",ncol(B))); names(origin) <- colnames(Z)
group  <- ifelse(LAB[colnames(Z)]==1,"Anthrax","Control")

## PANEL A — real gene-resampling bootstrap (not added noise)
cloud <- do.call(rbind, lapply(seq_len(NCLOUD), function(b){
  idx <- sample(nrow(Z), nrow(Z), replace=TRUE)
  mds <- cmdscale(dist(t(Z[idx,,drop=FALSE])), k=2)
  data.frame(Coord1=mds[,1],Coord2=mds[,2],Sample=rownames(mds),
             Dataset=origin[rownames(mds)],Group=group[rownames(mds)]) }))
pA <- ggplot(cloud,aes(Coord1,Coord2,colour=Group,shape=Dataset))+
  geom_point(size=1.1,alpha=.25)+
  stat_ellipse(aes(group=Sample),type="norm",linewidth=.3,colour="grey40")+
  scale_colour_manual(values=c(Control="#2E86C1",Anthrax="#C0392B"))+
  theme_classic(base_size=12)+
  labs(x="Coordinate 1",y="Coordinate 2",
       subtitle=paste(NCLOUD,"gene-resampling bootstrap replicates per sample"))
ggsave(paste0(FIG,"Fig_Bootstrap_PanelA_MDS.png"),pA,width=6.5,height=5.5,dpi=600,bg="white")

## PANEL B — multiscale bootstrap dendrogram
pv <- pvclust(Z, method.hclust="average", method.dist="correlation", nboot=NBOOT, parallel=FALSE)
png(paste0(FIG,"Fig_Bootstrap_PanelB_dendrogram.png"),width=9,height=6,units="in",res=600)
par(mar=c(9,4,3,2)); plot(pv,main="Bootstrap-resampled sample dendrogram (AU/BP %)",cex=.8,cex.pv=.7)
pvrect(pv,alpha=0.95); dev.off()
e <- pv$edges
boot <- data.frame(Cluster_edge=seq_len(nrow(e)),AU=round(e$au,4),BP=round(e$bp,4),
                   SE_AU=round(e$se.au,4),P_value=signif(1-e$au,3),
                   Supported_AU95=ifelse(e$au>=0.95,"Yes","No"))
write.csv(boot, paste0(TAB,"Table_bootstrap_confidence.csv"), row.names=FALSE); print(boot)

ph <- intersect(HUB, rownames(Z))
if (length(ph)>=3){
  pvh <- pvclust(Z[ph,,drop=FALSE],method.hclust="average",method.dist="correlation",
                 nboot=NBOOT,parallel=FALSE)
  png(paste0(SUP,"FigS4_bootstrap_hub_only.png"),width=9,height=6,units="in",res=600)
  par(mar=c(9,4,3,2)); plot(pvh,main="Bootstrap dendrogram — top-5 hub genes",cex=.8,cex.pv=.7)
  pvrect(pvh,alpha=0.95); dev.off()
  write.csv(data.frame(Cluster_edge=seq_len(nrow(pvh$edges)),AU=round(pvh$edges$au,4),
            BP=round(pvh$edges$bp,4)), paste0(SUP,"TableS8_bootstrap_hub_only.csv"), row.names=FALSE) }

imgA <- rasterGrob(readPNG(paste0(FIG,"Fig_Bootstrap_PanelA_MDS.png")),interpolate=TRUE)
imgB <- rasterGrob(readPNG(paste0(FIG,"Fig_Bootstrap_PanelB_dendrogram.png")),interpolate=TRUE)
lab <- function(g,t) arrangeGrob(g, top=textGrob(t,x=unit(.02,"npc"),hjust=0,
                                 gp=gpar(fontsize=20,fontface="bold")))
png(paste0(FIG,"Fig_Bootstrap_combined.png"),width=16,height=7,units="in",res=400)
grid.arrange(lab(imgA,"A"),lab(imgB,"B"),ncol=2,widths=c(1,1.1)); dev.off()
pdf(paste0(FIG,"Fig_Bootstrap_combined.pdf"),width=16,height=7)
grid.arrange(lab(imgA,"A"),lab(imgB,"B"),ncol=2,widths=c(1,1.1)); dev.off()
save(pv,boot,Z,origin,group,file=paste0(RDA,"Bootstrap_results.RData"))
cat("\\nBootstrap complete -> ",FINAL,"\\n")
''')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
features: 16 of 16 
Bootstrap (r = 0.5)... Done.
Bootstrap (r = 0.56)... Done.
Bootstrap (r = 0.69)... Done.
Bootstrap (r = 0.75)... Done.
Bootstrap (r = 0.88)... Done.
Bootstrap (r = 1.0)... Done.
Bootstrap (r = 1.06)... Done.
Bootstrap (r = 1.19)... Done.
Bootstrap (r = 1.25)... Done.
Bootstrap (r = 1.38)... Done.
   Cluster_edge     AU     BP  SE_AU  P_value Supported_AU95
1             1 0.9991 0.9644 0.0004 0.000920            Yes
2             2 0.9996 0.9860 0.0002 0.000388            Yes
3             3 0.8626 0.5720 0.0169 0.137000             No
4             4 0.9896 0.6997 0.0022 0.010400            Yes
5             5 0.9601 0.5609 0.0067 0.039900            Yes
6             6 0.9772 0.7436 0.0044 0.022800            Yes
7             7 0.7822 0.4211 0.0229 0.218000             No
8             8 0.8505 0.4647 0.0181 0.149000             No
9   

#Final1

In [ ]:
# ============ ANTHRAX BOOTSTRAP — FINAL (single cell, own folder) ============
from google.colab import drive; drive.mount('/content/drive')
import subprocess
subprocess.run("apt-get install -y r-base r-base-dev libcurl4-openssl-dev libssl-dev libxml2-dev",
               shell=True, capture_output=True)
subprocess.run("pip install -q rpy2", shell=True)
import rpy2.robjects as ro

ro.r('''
for (p in c("pvclust","data.table","ggplot2","png","gridExtra"))
  if (!requireNamespace(p, quietly=TRUE)) install.packages(p, repos="https://cloud.r-project.org")
suppressPackageStartupMessages({library(data.table);library(pvclust);library(ggplot2)
  library(grid);library(gridExtra);library(png)})
options(stringsAsFactors=FALSE)

## ------------------------------ CONFIG ------------------------------
ROOT <- "/content/drive/MyDrive/research/Anthrax/Anthraxpaper/"
EV   <- paste0(ROOT,"External_Validation/")
DATA <- paste0(ROOT,"Datasets/")
RES  <- paste0(ROOT,"Final/Bootstrap_Final/")
FIG  <- paste0(RES,"Figures/"); TAB <- paste0(RES,"Tables/")
SUP  <- paste0(RES,"Supplementary/"); RDA <- paste0(RES,"RData/")
for (d in c(RES,FIG,TAB,SUP,RDA)) dir.create(d, recursive=TRUE, showWarnings=FALSE)

LAB <- c(GSM359332=0,GSM359753=0,GSM359754=0,GSM359755=0,GSM359758=1,GSM359759=1,GSM359760=1,
         GSM305784=0,GSM305785=0,GSM305786=1,GSM305787=1)
COMMON <- c("GULP1","GATAD2A","ACSS1","LOC100506530","CNTN4","LINC00485",
            "WNT6","LOC101928461","IL20RA","VWC2","RNF128","TTC6")
HUB  <- c("WNT6","FZD9","DKK4","FZD6","FZD3")
FEAT <- unique(c(COMMON,HUB))
NBOOT <- 1000; NCLOUD <- 500; SEED <- 1

## ------------------------------ READERS ------------------------------
read_filtered <- function(path){
  df <- as.data.frame(fread(path))
  sym <- grep("^Gene[._ ]?symbol$|^Symbol$", colnames(df), ignore.case=TRUE, value=TRUE)[1]
  gsm <- grep("^GSM[0-9]+", colnames(df), value=TRUE)
  if (is.na(sym) || !length(gsm)) return(NULL)
  d <- df[, c(sym,gsm)]; s <- trimws(as.character(d[[sym]]))
  k <- s!="" & !is.na(s) & s!="NA" & !grepl("///", s, fixed=TRUE)
  d <- d[k,]; s <- s[k]; d[gsm] <- lapply(d[gsm], as.numeric)
  sm <- rowsum(as.matrix(d[gsm]), s); as.data.frame(sm / as.vector(table(s)[rownames(sm)]))
}
read_plain <- function(path){
  if (!file.exists(path)) return(NULL)
  dt <- fread(path); rn <- dt[[1]]; dt[[1]] <- NULL
  m <- as.data.frame(dt); rownames(m) <- make.unique(as.character(rn))
  m[] <- lapply(m, function(v) suppressWarnings(as.numeric(v))); m
}

PIPE <- list(
  P1 = list(tag="P1_filtered_data",
            A=function() read_filtered(paste0(EV,"Dis_GSE12131/filtered_data_GSE12131.csv")),
            B=function() read_filtered(paste0(EV,"Dis_GSE14390/filtered_data_GSE14390.csv"))),
  P2 = list(tag="P2_datasets_csv",
            A=function() read_plain(paste0(DATA,"GSE12131_gene_expression.csv")),
            B=function() read_plain(paste0(DATA,"GSE14390_gene_expression.csv")))
)

## ------------------------------ CORE ------------------------------
run_pipe <- function(p){
  A <- p$A(); B <- p$B()
  if (is.null(A) || is.null(B)) { cat("  [",p$tag,"] input missing - skipped\\n"); return(NULL) }
  if (max(A,na.rm=TRUE) > 50) A <- log2(A+1)
  if (max(B,na.rm=TRUE) > 50) B <- log2(B+1)
  feat <- intersect(FEAT, intersect(rownames(A), rownames(B)))
  M <- cbind(A[feat,,drop=FALSE], B[feat,,drop=FALSE])
  Z <- t(scale(t(M))); Z[is.na(Z)] <- 0
  org <- c(rep("GSE12131",ncol(A)),rep("GSE14390",ncol(B))); names(org) <- colnames(Z)
  grp <- ifelse(LAB[colnames(Z)]==1,"Anthrax","Control")
  cat(sprintf("  [%s] %d features | %d samples (%d anthrax / %d control)\\n",
              p$tag, length(feat), ncol(Z), sum(grp=="Anthrax"), sum(grp=="Control")))
  set.seed(SEED)
  pv <- pvclust(Z, method.hclust="average", method.dist="correlation",
                nboot=NBOOT, parallel=FALSE, quiet=TRUE)
  e <- pv$edges
  tb <- data.frame(Pipeline=p$tag, Cluster_edge=seq_len(nrow(e)),
                   AU=round(e$au,4), BP=round(e$bp,4), SE_AU=round(e$se.au,4),
                   P_value=signif(1-e$au,3),
                   Supported_AU95=ifelse(e$au>=0.95,"Yes","No"))
  tb$Is_root <- c(rep("No", nrow(e)-1), "Yes")          # root edge trivially AU=1
  list(pv=pv, Z=Z, origin=org, group=grp, tab=tb, feat=feat)
}

cat("Running bootstrap under both preprocessing pipelines...\\n")
R <- lapply(PIPE, run_pipe); R <- R[!sapply(R, is.null)]
stopifnot(length(R) >= 1)

allTab <- do.call(rbind, lapply(R, function(x) x$tab))
write.csv(allTab, paste0(TAB,"Table_bootstrap_confidence.csv"), row.names=FALSE)
cat("\\n--- bootstrap confidence (both pipelines) ---\\n"); print(allTab, row.names=FALSE)

## ---- preprocessing sensitivity: একই edge দুই pipeline-এ কতটা বদলায় ----
if (length(R) == 2){
  a <- R[[1]]$tab; b <- R[[2]]$tab
  n <- min(nrow(a), nrow(b))
  sens <- data.frame(Cluster_edge=1:n, AU_P1=a$AU[1:n], AU_P2=b$AU[1:n],
                     Delta_AU=round(abs(a$AU[1:n]-b$AU[1:n]),4),
                     Support_P1=a$Supported_AU95[1:n], Support_P2=b$Supported_AU95[1:n])
  sens$Agreement <- ifelse(sens$Support_P1==sens$Support_P2,"consistent","FLIPPED")
  write.csv(sens, paste0(SUP,"TableS_preprocessing_sensitivity.csv"), row.names=FALSE)
  cat("\\n--- preprocessing sensitivity ---\\n"); print(sens, row.names=FALSE)
  cat(sprintf("\\n>>> edges flipping support between pipelines: %d of %d | max |dAU| = %.3f\\n",
              sum(sens$Agreement=="FLIPPED"), n, max(sens$Delta_AU)))
}

## ---- MAIN pipeline (P1) for figures ----
m <- R[[1]]; pv <- m$pv; Z <- m$Z; origin <- m$origin; group <- m$group

## ---- cluster কী আলাদা করছে: dataset নাকি group ----
k2 <- cutree(pv$hclust, k=2)
cmp <- data.frame(Sample=names(k2), Cluster=as.integer(k2),
                  Dataset=origin[names(k2)], Group=group[names(k2)])
agr <- function(t) max(sum(diag(t)), sum(t)-sum(diag(t)))/sum(t)
aD <- agr(table(cmp$Cluster, cmp$Dataset)); aG <- agr(table(cmp$Cluster, cmp$Group))
DRIVER <- if (aD > aG) "BATCH (dataset)" else if (aG > aD) "BIOLOGY (group)" else "AMBIGUOUS"
write.csv(cmp, paste0(SUP,"TableS_cluster_membership.csv"), row.names=FALSE)
cat("\\n--- 2-cluster solution ---\\n"); print(cmp, row.names=FALSE)
cat("\\nCluster x DATASET:\\n"); print(table(cmp$Cluster, cmp$Dataset))
cat("\\nCluster x GROUP:\\n");   print(table(cmp$Cluster, cmp$Group))
cat(sprintf("\\n>>> agreement with DATASET = %.2f | with GROUP = %.2f  -> driven by %s\\n",
            aD, aG, DRIVER))

## ---- PANEL A: real gene-resampling bootstrap MDS ----
set.seed(SEED)
cloud <- do.call(rbind, lapply(seq_len(NCLOUD), function(b){
  idx <- sample(nrow(Z), nrow(Z), replace=TRUE)
  d <- dist(t(Z[idx,,drop=FALSE]))
  if (any(!is.finite(d))) return(NULL)
  mds <- cmdscale(d, k=2)
  data.frame(Coord1=mds[,1],Coord2=mds[,2],Sample=rownames(mds),
             Dataset=origin[rownames(mds)],Group=group[rownames(mds)]) }))
pA <- ggplot(cloud,aes(Coord1,Coord2,colour=Group,shape=Dataset))+
  geom_point(size=1.1,alpha=.25)+
  stat_ellipse(aes(group=Sample),type="norm",linewidth=.3,colour="grey40")+
  scale_colour_manual(values=c(Control="#2E86C1",Anthrax="#C0392B"))+
  theme_classic(base_size=12)+
  labs(x="Coordinate 1",y="Coordinate 2",
       subtitle=paste0(NCLOUD," gene-resampling bootstrap replicates per sample"))
ggsave(paste0(FIG,"Fig_Bootstrap_PanelA_MDS.png"),pA,width=6.5,height=5.5,dpi=600,bg="white")

## ---- PANEL B: dendrogram with AU/BP ----
png(paste0(FIG,"Fig_Bootstrap_PanelB_dendrogram.png"),width=9,height=6,units="in",res=600)
par(mar=c(9,4,3,2))
plot(pv, main=sprintf("Bootstrap sample dendrogram (AU/BP %%) — driven by %s", DRIVER),
     cex=.8, cex.pv=.7)
pvrect(pv, alpha=0.95); dev.off()

## ---- hub-only (guarded) ----
ph <- intersect(HUB, rownames(Z)); hub_ok <- FALSE
if (length(ph) >= 3){
  set.seed(SEED+1)
  pvh <- tryCatch(suppressWarnings(
    pvclust(Z[ph,,drop=FALSE], method.hclust="average", method.dist="correlation",
            nboot=NBOOT, parallel=FALSE, quiet=TRUE)), error=function(e) NULL)
  if (!is.null(pvh) && all(is.finite(pvh$edges$au)) && sd(pvh$edges$au) > 1e-6){
    hub_ok <- TRUE
    png(paste0(SUP,"FigS_bootstrap_hub_only.png"),width=9,height=6,units="in",res=600)
    par(mar=c(9,4,3,2)); plot(pvh,main="Bootstrap dendrogram — top-5 hub genes",cex=.8,cex.pv=.7)
    pvrect(pvh,alpha=0.95); dev.off()
    write.csv(data.frame(Cluster_edge=seq_len(nrow(pvh$edges)),
      AU=round(pvh$edges$au,4), BP=round(pvh$edges$bp,4)),
      paste0(SUP,"TableS_bootstrap_hub_only.csv"), row.names=FALSE)
  } else cat("\\n>>> hub-only bootstrap DEGENERATE (5 genes) - not reported\\n")
} else cat("\\n>>> fewer than 3 hub genes present - hub-only skipped\\n")

## ---- combined figure ----
iA <- rasterGrob(readPNG(paste0(FIG,"Fig_Bootstrap_PanelA_MDS.png")),interpolate=TRUE)
iB <- rasterGrob(readPNG(paste0(FIG,"Fig_Bootstrap_PanelB_dendrogram.png")),interpolate=TRUE)
lb <- function(g,t) arrangeGrob(g, top=textGrob(t,x=unit(.02,"npc"),hjust=0,
                                gp=gpar(fontsize=20,fontface="bold")))
for (dev_ in list(
  function() png(paste0(FIG,"Fig_Bootstrap_combined.png"),width=16,height=7,units="in",res=400),
  function() pdf(paste0(FIG,"Fig_Bootstrap_combined.pdf"),width=16,height=7))){
  dev_(); grid.arrange(lb(iA,"A"), lb(iB,"B"), ncol=2, widths=c(1,1.1)); dev.off() }

## ---- validity report ----
nonroot <- subset(m$tab, Is_root=="No")
rep <- data.frame(
  Item=c("Datasets","Samples (total)","Anthrax / Control","Panel genes used",
         "Bootstrap replicates","Internal nodes (excl. root)","Nodes AU >= 0.95",
         "Lowest AU","Pipelines compared","Edges flipping support",
         "Cluster agreement: dataset","Cluster agreement: group",
         "2-cluster split driven by","Hub-only (5 genes) usable","Overall interpretation"),
  Value=c("GSE12131 + GSE14390", ncol(Z),
          paste0(sum(group=="Anthrax")," / ",sum(group=="Control")),
          paste0(length(m$feat)," of ",length(FEAT)), NBOOT, nrow(nonroot),
          sum(nonroot$Supported_AU95=="Yes"), round(min(nonroot$AU),3),
          length(R),
          if (exists("sens")) sum(sens$Agreement=="FLIPPED") else NA,
          round(aD,2), round(aG,2), DRIVER,
          ifelse(hub_ok,"Yes","No (degenerate)"),
          ifelse(DRIVER=="BIOLOGY (group)" &&
                 sum(nonroot$Supported_AU95=="Yes") >= ceiling(nrow(nonroot)/2),
                 "Usable as clustering-stability support",
                 "Report as stability check only - NOT validation")))
write.csv(rep, paste0(SUP,"TableS_bootstrap_validity_report.csv"), row.names=FALSE)
cat("\\n================ VALIDITY REPORT ================\\n"); print(rep, right=FALSE)

save(R, pv, Z, origin, group, cmp, aD, aG, DRIVER, rep,
     file=paste0(RDA,"Bootstrap_Final.RData"))
cat("\\nSaved -> ",RES,"\\n")
''')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running bootstrap under both preprocessing pipelines...
  [P1_filtered_data] 16 features | 11 samples (5 anthrax / 6 control)
  [P2_datasets_csv] 16 features | 11 samples (5 anthrax / 6 control)

--- bootstrap confidence (both pipelines) ---
         Pipeline Cluster_edge     AU     BP  SE_AU P_value Supported_AU95
 P1_filtered_data            1 0.7315 0.6638 0.0258  0.2690             No
 P1_filtered_data            2 0.9576 0.7864 0.0075  0.0424            Yes
 P1_filtered_data            3 0.9623 0.5636 0.0064  0.0377            Yes
 P1_filtered_data            4 0.9191 0.6925 0.0119  0.0809             No
 P1_filtered_data            5 0.8559 0.5428 0.0175  0.1440             No
 P1_filtered_data            6 0.7979 0.4753 0.0217  0.2020             No
 P1_filtered_data            7 0.7683 0.4745 0.0234  0.2320             No
 P1_filtered_data            

In [ ]:
# ======== ANTHRAX BOOTSTRAP — FINAL + PER-HUB-GENE (single cell) ========
from google.colab import drive; drive.mount('/content/drive')
import subprocess
subprocess.run("apt-get install -y r-base r-base-dev libcurl4-openssl-dev libssl-dev libxml2-dev",
               shell=True, capture_output=True)
subprocess.run("pip install -q rpy2", shell=True)
import rpy2.robjects as ro

ro.r('''
for (p in c("pvclust","data.table","ggplot2","png","gridExtra","patchwork","dendextend"))
  if (!requireNamespace(p, quietly=TRUE)) install.packages(p, repos="https://cloud.r-project.org")
suppressPackageStartupMessages({library(data.table);library(pvclust);library(ggplot2)
  library(grid);library(gridExtra);library(png);library(patchwork);library(dendextend)})
options(stringsAsFactors=FALSE)

## ============================ CONFIG ============================
ROOT <- "/content/drive/MyDrive/research/Anthrax/Anthraxpaper/"
EV   <- paste0(ROOT,"External_Validation/")
DATA <- paste0(ROOT,"Datasets/")
RES  <- paste0(ROOT,"Final/Bootstrap_Final/")
FIG  <- paste0(RES,"Figures/");           TAB <- paste0(RES,"Tables/")
GFIG <- paste0(RES,"Figures/HubGenes/");  SUP <- paste0(RES,"Supplementary/")
RDA  <- paste0(RES,"RData/")
for (d in c(RES,FIG,GFIG,TAB,SUP,RDA)) dir.create(d, recursive=TRUE, showWarnings=FALSE)

LAB <- c(GSM359332=0,GSM359753=0,GSM359754=0,GSM359755=0,GSM359758=1,GSM359759=1,GSM359760=1,
         GSM305784=0,GSM305785=0,GSM305786=1,GSM305787=1)
COMMON <- c("GULP1","GATAD2A","ACSS1","LOC100506530","CNTN4","LINC00485",
            "WNT6","LOC101928461","IL20RA","VWC2","RNF128","TTC6")
HUB  <- c("WNT6","FZD9","DKK4","FZD6","FZD3")          # CytoHubba degree rank 1-5
FEAT <- unique(c(COMMON,HUB))
NBOOT <- 1000; NCLOUD <- 500; NBOOT_CI <- 5000; SEED <- 1
COLG <- c(Control="#2E86C1", Anthrax="#C0392B")

## ============================ READERS ============================
read_filtered <- function(path){
  if (!file.exists(path)) return(NULL)
  df <- as.data.frame(fread(path))
  sym <- grep("^Gene[._ ]?symbol$|^Symbol$", colnames(df), ignore.case=TRUE, value=TRUE)[1]
  gsm <- grep("^GSM[0-9]+", colnames(df), value=TRUE)
  if (is.na(sym) || !length(gsm)) return(NULL)
  d <- df[, c(sym,gsm)]; s <- trimws(as.character(d[[sym]]))
  k <- s!="" & !is.na(s) & s!="NA" & !grepl("///", s, fixed=TRUE)
  d <- d[k,]; s <- s[k]; d[gsm] <- lapply(d[gsm], as.numeric)
  sm <- rowsum(as.matrix(d[gsm]), s); as.data.frame(sm / as.vector(table(s)[rownames(sm)]))
}
read_plain <- function(path){
  if (!file.exists(path)) return(NULL)
  dt <- fread(path); rn <- dt[[1]]; dt[[1]] <- NULL
  m <- as.data.frame(dt); rownames(m) <- make.unique(as.character(rn))
  m[] <- lapply(m, function(v) suppressWarnings(as.numeric(v))); m
}
## per-dataset scale decision (দুই platform আলাদা করে দেখা হয়)
fix_scale <- function(M, nm){
  if (min(M,na.rm=TRUE) >= 0 && max(M,na.rm=TRUE) > 50){
    cat("   ",nm,": log2(x+1) applied (raw intensity)\\n"); return(log2(M+1)) }
  cat("   ",nm,": no transform (already log/normalized)\\n"); M
}

PIPE <- list(
  P1 = list(tag="P1_filtered_data",
            A=function() read_filtered(paste0(EV,"Dis_GSE12131/filtered_data_GSE12131.csv")),
            B=function() read_filtered(paste0(EV,"Dis_GSE14390/filtered_data_GSE14390.csv"))),
  P2 = list(tag="P2_datasets_csv",
            A=function() read_plain(paste0(DATA,"GSE12131_gene_expression.csv")),
            B=function() read_plain(paste0(DATA,"GSE14390_gene_expression.csv")))
)

## ============================ PART 1: PANEL BOOTSTRAP ============================
run_pipe <- function(p){
  A <- p$A(); B <- p$B()
  if (is.null(A) || is.null(B)) { cat("  [",p$tag,"] input missing - skipped\\n"); return(NULL) }
  cat("  [",p$tag,"]\\n"); A <- fix_scale(A,"GSE12131"); B <- fix_scale(B,"GSE14390")
  feat <- intersect(FEAT, intersect(rownames(A), rownames(B)))
  M <- cbind(A[feat,,drop=FALSE], B[feat,,drop=FALSE])
  Z <- t(scale(t(M))); Z[is.na(Z)] <- 0
  org <- c(rep("GSE12131",ncol(A)),rep("GSE14390",ncol(B))); names(org) <- colnames(Z)
  grp <- ifelse(LAB[colnames(Z)]==1,"Anthrax","Control")
  cat(sprintf("    -> %d features | %d samples (%d anthrax / %d control)\\n",
              length(feat), ncol(Z), sum(grp=="Anthrax"), sum(grp=="Control")))
  set.seed(SEED)
  pv <- pvclust(Z, method.hclust="average", method.dist="correlation",
                nboot=NBOOT, parallel=FALSE, quiet=TRUE)
  e <- pv$edges
  tb <- data.frame(Pipeline=p$tag, Cluster_edge=seq_len(nrow(e)),
                   AU=round(e$au,4), BP=round(e$bp,4), SE_AU=round(e$se.au,4),
                   P_value=signif(1-e$au,3),
                   Supported_AU95=ifelse(e$au>=0.95,"Yes","No"),
                   Is_root=c(rep("No",nrow(e)-1),"Yes"))
  list(pv=pv, Z=Z, origin=org, group=grp, tab=tb, feat=feat, A=A, B=B)
}
cat("Running panel bootstrap under both preprocessing pipelines...\\n")
R <- lapply(PIPE, run_pipe); R <- R[!sapply(R, is.null)]
stopifnot(length(R) >= 1)

allTab <- do.call(rbind, lapply(R, function(x) x$tab))
write.csv(allTab, paste0(TAB,"Table_bootstrap_confidence.csv"), row.names=FALSE)
cat("\\n--- bootstrap confidence ---\\n"); print(allTab, row.names=FALSE)

if (length(R) == 2){
  a <- R[[1]]$tab; b <- R[[2]]$tab; n <- min(nrow(a),nrow(b))
  sens <- data.frame(Cluster_edge=1:n, AU_P1=a$AU[1:n], AU_P2=b$AU[1:n],
                     Delta_AU=round(abs(a$AU[1:n]-b$AU[1:n]),4),
                     Support_P1=a$Supported_AU95[1:n], Support_P2=b$Supported_AU95[1:n])
  sens$Agreement <- ifelse(sens$Support_P1==sens$Support_P2,"consistent","FLIPPED")
  write.csv(sens, paste0(SUP,"TableS_preprocessing_sensitivity.csv"), row.names=FALSE)
  cat("\\n--- preprocessing sensitivity ---\\n"); print(sens, row.names=FALSE)
  cat(sprintf("\\n>>> edges flipping support: %d of %d | max |dAU| = %.3f\\n",
              sum(sens$Agreement=="FLIPPED"), n, max(sens$Delta_AU)))
}

m <- R[[1]]; pv <- m$pv; Z <- m$Z; origin <- m$origin; group <- m$group
k2 <- cutree(pv$hclust, k=2)
cmp <- data.frame(Sample=names(k2), Cluster=as.integer(k2),
                  Dataset=origin[names(k2)], Group=group[names(k2)])
agr <- function(t) max(sum(diag(t)), sum(t)-sum(diag(t)))/sum(t)
aD <- agr(table(cmp$Cluster,cmp$Dataset)); aG <- agr(table(cmp$Cluster,cmp$Group))
DRIVER <- if (aD>aG) "BATCH (dataset)" else if (aG>aD) "BIOLOGY (group)" else "AMBIGUOUS"
write.csv(cmp, paste0(SUP,"TableS_cluster_membership.csv"), row.names=FALSE)
cat("\\n--- 2-cluster solution ---\\n"); print(cmp, row.names=FALSE)
cat("\\nCluster x GROUP:\\n"); print(table(cmp$Cluster, cmp$Group))
cat(sprintf("\\n>>> agreement DATASET = %.2f | GROUP = %.2f  -> %s\\n", aD, aG, DRIVER))

## ---- PANEL A (clearer: bigger points, per-sample centroid labels) ----
set.seed(SEED)
cloud <- do.call(rbind, lapply(seq_len(NCLOUD), function(b){
  idx <- sample(nrow(Z), nrow(Z), replace=TRUE)
  d <- dist(t(Z[idx,,drop=FALSE])); if (any(!is.finite(d))) return(NULL)
  mds <- cmdscale(d, k=2)
  data.frame(Coord1=mds[,1],Coord2=mds[,2],Sample=rownames(mds),
             Dataset=origin[rownames(mds)],Group=group[rownames(mds)]) }))
cen <- aggregate(cbind(Coord1,Coord2) ~ Sample+Group+Dataset, cloud, mean)
pA <- ggplot(cloud, aes(Coord1,Coord2,colour=Group))+
  geom_point(aes(shape=Dataset), size=1.3, alpha=.22)+
  stat_ellipse(aes(group=Sample), type="norm", linewidth=.45, colour="grey35")+
  geom_point(data=cen, aes(shape=Dataset), size=4.2, stroke=1.1, fill="white")+
  geom_text(data=cen, aes(label=Sample), vjust=-1.25, size=3.3, fontface="bold", show.legend=FALSE)+
  scale_colour_manual(values=COLG)+ scale_shape_manual(values=c(21,24))+
  theme_classic(base_size=14)+
  theme(legend.position="bottom", legend.title=element_blank(),
        axis.text=element_text(colour="black"))+
  labs(x="Coordinate 1", y="Coordinate 2",
       subtitle=paste0(NCLOUD," gene-resampling bootstrap replicates per sample"))
ggsave(paste0(FIG,"Fig_Bootstrap_PanelA_MDS.png"), pA, width=7.5, height=6.5, dpi=600, bg="white")
ggsave(paste0(FIG,"Fig_Bootstrap_PanelA_MDS.pdf"), pA, width=7.5, height=6.5, bg="white")

## ---- PANEL B (clearer: leaf labels coloured by group, AU only, bigger text) ----
draw_dendro <- function(){
  par(mar=c(10,5,4,2), xpd=NA)
  hc <- pv$hclust
  lv <- group[hc$labels[hc$order]]
  plot(hc, main=sprintf("Bootstrap sample dendrogram  (AU %%)\\nsplit driven by %s", DRIVER),
       sub="", xlab="", ylab="Height", hang=-1, cex=1.05, cex.main=1.15, cex.lab=1.05)
  ## leaf label colour strip
  pos <- seq_along(hc$order)
  points(pos, rep(par("usr")[3] + diff(par("usr")[3:4])*0.012, length(pos)),
         pch=15, cex=2.1, col=COLG[lv])
  ## AU labels at nodes
  A <- pv$hclust$merge; nn <- nrow(A)
  xax <- numeric(nn); zz <- match(seq_len(nn+1), hc$order)
  for (i in seq_len(nn)){
    v <- c(if (A[i,1]<0) zz[-A[i,1]] else xax[A[i,1]],
           if (A[i,2]<0) zz[-A[i,2]] else xax[A[i,2]])
    xax[i] <- mean(v) }
  au <- round(pv$edges$au*100)
  text(xax, hc$height, labels=au, pos=3, offset=.35, cex=1.0, font=2,
       col=ifelse(pv$edges$au>=0.95, "#C0392B", "grey30"))
  legend("topright", legend=c("Control","Anthrax","AU >= 95","AU < 95"),
         pch=c(15,15,NA,NA), lty=c(NA,NA,1,1), lwd=c(NA,NA,2,2),
         col=c(COLG["Control"],COLG["Anthrax"],"#C0392B","grey30"),
         bty="n", cex=.95)
}
png(paste0(FIG,"Fig_Bootstrap_PanelB_dendrogram.png"), width=10, height=7, units="in", res=600)
draw_dendro(); dev.off()
pdf(paste0(FIG,"Fig_Bootstrap_PanelB_dendrogram.pdf"), width=10, height=7); draw_dendro(); dev.off()

## ---- hub-only dendrogram (guarded) ----
ph <- intersect(HUB, rownames(Z)); hub_ok <- FALSE
if (length(ph) >= 3){
  set.seed(SEED+1)
  pvh <- tryCatch(suppressWarnings(pvclust(Z[ph,,drop=FALSE], method.hclust="average",
          method.dist="correlation", nboot=NBOOT, parallel=FALSE, quiet=TRUE)),
          error=function(e) NULL)
  if (!is.null(pvh) && all(is.finite(pvh$edges$au)) && sd(pvh$edges$au) > 1e-6){
    hub_ok <- TRUE
    png(paste0(SUP,"FigS_bootstrap_hub_only.png"), width=10, height=7, units="in", res=600)
    par(mar=c(10,5,4,2)); plot(pvh, main="Bootstrap dendrogram - top-5 hub genes only",
                               cex=1.0, cex.pv=.85); pvrect(pvh, alpha=.95); dev.off()
    write.csv(data.frame(Cluster_edge=seq_len(nrow(pvh$edges)),
      AU=round(pvh$edges$au,4), BP=round(pvh$edges$bp,4)),
      paste0(SUP,"TableS_bootstrap_hub_only.csv"), row.names=FALSE)
  } else cat("\\n>>> hub-only bootstrap DEGENERATE (5 genes) - not reported\\n")
}

## ---- combined A+B ----
iA <- rasterGrob(readPNG(paste0(FIG,"Fig_Bootstrap_PanelA_MDS.png")), interpolate=TRUE)
iB <- rasterGrob(readPNG(paste0(FIG,"Fig_Bootstrap_PanelB_dendrogram.png")), interpolate=TRUE)
lb <- function(g,t) arrangeGrob(g, top=textGrob(t, x=unit(.02,"npc"), hjust=0,
                                gp=gpar(fontsize=22, fontface="bold")))
png(paste0(FIG,"Fig_Bootstrap_combined.png"), width=17, height=7.5, units="in", res=500)
grid.arrange(lb(iA,"A"), lb(iB,"B"), ncol=2, widths=c(1,1.15)); dev.off()
pdf(paste0(FIG,"Fig_Bootstrap_combined.pdf"), width=17, height=7.5)
grid.arrange(lb(iA,"A"), lb(iB,"B"), ncol=2, widths=c(1,1.15)); dev.off()

## ============== PART 2: PER-HUB-GENE BOOTSTRAP (by GROUP) ==============
cat("\\n================ PER-HUB-GENE BOOTSTRAP ================\\n")
A <- m$A; B <- m$B
zrow <- function(M){ t(apply(M,1,function(v){ s <- sd(v,na.rm=TRUE)
  if (!is.finite(s) || s==0) rep(0,length(v)) else (v-mean(v,na.rm=TRUE))/s })) }
gg <- intersect(HUB, intersect(rownames(A), rownames(B)))
cat("hub genes available:", length(gg), "of", length(HUB), "-", paste(gg,collapse=", "), "\\n")
ZG <- cbind(zrow(A[gg,,drop=FALSE]), zrow(B[gg,,drop=FALSE]))
colnames(ZG) <- c(colnames(A), colnames(B)); rownames(ZG) <- gg
grpG <- ifelse(LAB[colnames(ZG)]==1,"Anthrax","Control")
dstG <- c(rep("GSE12131",ncol(A)), rep("GSE14390",ncol(B)))

boot_ci <- function(x, seed){
  x <- x[is.finite(x)]; n <- length(x)
  if (n < 2) return(c(mean=ifelse(n==1,x,NA), lo=NA, hi=NA, n=n))
  set.seed(seed); bm <- replicate(NBOOT_CI, mean(sample(x, n, replace=TRUE)))
  c(mean=mean(x), lo=unname(quantile(bm,.025)), hi=unname(quantile(bm,.975)), n=n) }

hubRows <- list(); hubPlots <- list()
for (x in gg){
  d <- data.frame(Sample=colnames(ZG), Group=factor(grpG, levels=c("Control","Anthrax")),
                  Dataset=dstG, Expr=as.numeric(ZG[x,]))
  d <- d[is.finite(d$Expr),]
  cC <- boot_ci(d$Expr[d$Group=="Control"], 1); cA <- boot_ci(d$Expr[d$Group=="Anthrax"], 2)
  ci <- data.frame(Group=factor(c("Control","Anthrax"), levels=c("Control","Anthrax")),
                   Mean=c(cC["mean"],cA["mean"]), lo=c(cC["lo"],cA["lo"]),
                   hi=c(cC["hi"],cA["hi"]), N=c(cC["n"],cA["n"]))
  ovl <- !(ci$hi[1] < ci$lo[2] || ci$hi[2] < ci$lo[1])
  wt  <- suppressWarnings(wilcox.test(Expr ~ Group, data=d)$p.value)

  p1 <- ggplot(d, aes(Group, Expr, fill=Group))+
    geom_boxplot(alpha=.65, outlier.shape=NA, width=.55, linewidth=.6)+
    geom_jitter(aes(shape=Dataset), width=.11, size=3, alpha=.95, colour="black", stroke=.5)+
    scale_fill_manual(values=COLG)+ scale_shape_manual(values=c(21,24))+
    theme_bw(13)+theme(panel.grid.minor=element_blank(), panel.grid.major.x=element_blank(),
      legend.position="bottom", legend.title=element_blank(),
      plot.title=element_text(face="bold",hjust=.5), axis.text=element_text(colour="black"))+
    labs(title="A. Expression distribution", x="", y="Expression (z-score)")
  p2 <- ggplot(ci, aes(Group, Mean, fill=Group))+
    geom_hline(yintercept=0, colour="grey70", linewidth=.4)+
    geom_col(width=.55, colour="black", alpha=.85, linewidth=.6)+
    geom_errorbar(aes(ymin=lo, ymax=hi), width=.16, linewidth=.8, na.rm=TRUE)+
    geom_text(aes(label=paste0("n=",N), y=pmax(hi,Mean,0,na.rm=TRUE)),
              vjust=-.9, size=3.6, fontface="italic")+
    scale_fill_manual(values=COLG)+ theme_bw(13)+
    theme(panel.grid.minor=element_blank(), panel.grid.major.x=element_blank(),
      legend.position="none", plot.title=element_text(face="bold",hjust=.5),
      axis.text=element_text(colour="black"))+
    labs(title=paste0("B. Bootstrap 95% CI", if (!ovl) "  (non-overlapping)" else ""),
         x="", y="Mean z-score")+
    expand_limits(y=c(min(ci$lo,0,na.rm=TRUE)*1.25, max(ci$hi,0,na.rm=TRUE)*1.25))
  pp <- (p1|p2) + plot_annotation(
    title=x, subtitle=sprintf("CI overlap: %s   |   Wilcoxon p = %.3f   |   %s",
      ifelse(ovl,"yes","NO"), wt,
      ifelse(ci$Mean[2]>ci$Mean[1],"higher in anthrax","lower in anthrax")),
    theme=theme(plot.title=element_text(face="bold.italic",size=17,hjust=.5),
                plot.subtitle=element_text(size=10.5,hjust=.5,colour="grey25")))
  hubPlots[[x]] <- pp
  ggsave(paste0(GFIG,x,"_bootstrap.png"), pp, width=9, height=5, dpi=600, bg="white")
  ggsave(paste0(GFIG,x,"_bootstrap.pdf"), pp, width=9, height=5, bg="white")
  cat(sprintf("  %-6s  control %.3f [%.3f, %.3f] | anthrax %.3f [%.3f, %.3f] | overlap %s | p=%.3f\\n",
      x, ci$Mean[1],ci$lo[1],ci$hi[1], ci$Mean[2],ci$lo[2],ci$hi[2],
      ifelse(ovl,"yes","NO"), wt))
  hubRows[[x]] <- data.frame(Gene=x, Group=ci$Group, N=ci$N, Mean_z=round(ci$Mean,4),
    CI_low=round(ci$lo,4), CI_high=round(ci$hi,4), CI_overlap=ifelse(ovl,"Yes","No"),
    Wilcoxon_p=round(wt,4),
    Direction=ifelse(ci$Mean[2]>ci$Mean[1],"Higher in anthrax","Lower in anthrax"))
}
hubS <- do.call(rbind, hubRows); rownames(hubS) <- NULL
write.csv(hubS, paste0(TAB,"Table_hubgene_bootstrapCI_by_group.csv"), row.names=FALSE)
cat("\\n--- hub gene bootstrap summary ---\\n"); print(hubS, row.names=FALSE)
sepg <- unique(hubS$Gene[hubS$CI_overlap=="No"])
cat("\\n>>> hub genes with NON-overlapping 95% CI:",
    if (length(sepg)) paste(sepg, collapse=", ") else "none", "\\n")

ov <- wrap_plots(hubPlots, ncol=1) + plot_annotation(
  title="Hub gene bootstrap analysis (Anthrax vs Control, z-scored within dataset)",
  theme=theme(plot.title=element_text(face="bold", size=19, hjust=.5)))
ggsave(paste0(GFIG,"AllHubGenes_overview.png"), ov, width=9.5, height=5*length(hubPlots),
       dpi=400, bg="white", limitsize=FALSE)
ggsave(paste0(GFIG,"AllHubGenes_overview.pdf"), ov, width=9.5, height=5*length(hubPlots),
       bg="white", limitsize=FALSE)

## ============================ VALIDITY REPORT ============================
nonroot <- subset(m$tab, Is_root=="No")
rep <- data.frame(
  Item=c("Datasets","Samples (total)","Anthrax / Control","Panel genes used",
         "Bootstrap replicates (panel)","Bootstrap replicates (per-gene CI)",
         "Internal nodes (excl. root)","Nodes AU >= 0.95","Lowest AU",
         "Pipelines compared","Edges flipping support","Cluster agreement: dataset",
         "Cluster agreement: group","2-cluster split driven by",
         "Hub-only dendrogram usable","Hub genes with non-overlapping CI",
         "Overall interpretation"),
  Value=c("GSE12131 + GSE14390", ncol(Z),
          paste0(sum(group=="Anthrax")," / ",sum(group=="Control")),
          paste0(length(m$feat)," of ",length(FEAT)), NBOOT, NBOOT_CI,
          nrow(nonroot), sum(nonroot$Supported_AU95=="Yes"), round(min(nonroot$AU),3),
          length(R), if (exists("sens")) sum(sens$Agreement=="FLIPPED") else NA,
          round(aD,2), round(aG,2), DRIVER, ifelse(hub_ok,"Yes","No (degenerate)"),
          ifelse(length(sepg), paste(sepg,collapse=", "), "none"),
          ifelse(DRIVER=="BIOLOGY (group)" &&
                 sum(nonroot$Supported_AU95=="Yes") >= ceiling(nrow(nonroot)/2),
                 "Usable as clustering-stability support",
                 "Report as stability check only - NOT validation")))
write.csv(rep, paste0(SUP,"TableS_bootstrap_validity_report.csv"), row.names=FALSE)
cat("\\n================ VALIDITY REPORT ================\\n"); print(rep, right=FALSE)

save(R, pv, Z, origin, group, cmp, aD, aG, DRIVER, hubS, ZG, rep,
     file=paste0(RDA,"Bootstrap_Final.RData"))
cat("\\nSaved -> ", RES, "\\n")
''')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


(as ‘lib’ is unspecified)







	‘/tmp/Rtmppub1Jx/downloaded_packages’

(as ‘lib’ is unspecified)






	‘/tmp/Rtmppub1Jx/downloaded_packages’



Running panel bootstrap under both preprocessing pipelines...
  [ P1_filtered_data ]
    GSE12131 : log2(x+1) applied (raw intensity)
    GSE14390 : no transform (already log/normalized)
    -> 16 features | 11 samples (5 anthrax / 6 control)
  [ P2_datasets_csv ]
    GSE12131 : log2(x+1) applied (raw intensity)
    GSE14390 : no transform (already log/normalized)
    -> 16 features | 11 samples (5 anthrax / 6 control)

--- bootstrap confidence ---
         Pipeline Cluster_edge     AU     BP  SE_AU P_value Supported_AU95
 P1_filtered_data            1 0.7315 0.6638 0.0258  0.2690             No
 P1_filtered_data            2 0.9576 0.7864 0.0075  0.0424            Yes
 P1_filtered_data            3 0.9623 0.5636 0.0064  0.0377            Yes
 P1_filtered_data            4 0.9191 0.6925 0.0119  0.0809             No
 P1_filtered_data            5 0.8559 0.5428 0.0175  0.1440             No
 P1_filtered_data            6 0.7979 0.4753 0.0217  0.2020             No
 P1_filtered_data     

In [ ]:
# ====== ANTHRAX BOOTSTRAP — FINAL (clean publication figures) ======
from google.colab import drive; drive.mount('/content/drive')
import subprocess
subprocess.run("apt-get install -y r-base r-base-dev libcurl4-openssl-dev libssl-dev libxml2-dev",
               shell=True, capture_output=True)
subprocess.run("pip install -q rpy2", shell=True)
import rpy2.robjects as ro

ro.r('''
for (p in c("pvclust","data.table","ggplot2","png","gridExtra","patchwork","ggrepel","ggdendro"))
  if (!requireNamespace(p, quietly=TRUE)) install.packages(p, repos="https://cloud.r-project.org")
suppressPackageStartupMessages({library(data.table);library(pvclust);library(ggplot2)
  library(grid);library(gridExtra);library(png);library(patchwork);library(ggrepel);library(ggdendro)})
options(stringsAsFactors=FALSE)

## ============================ CONFIG ============================
ROOT <- "/content/drive/MyDrive/research/Anthrax/Anthraxpaper/"
EV   <- paste0(ROOT,"External_Validation/"); DATA <- paste0(ROOT,"Datasets/")
RES  <- paste0(ROOT,"Final/Bootstrap_Final/")
FIG  <- paste0(RES,"Figures/"); GFIG <- paste0(RES,"Figures/HubGenes/")
TAB  <- paste0(RES,"Tables/");  SUP <- paste0(RES,"Supplementary/"); RDA <- paste0(RES,"RData/")
for (d in c(RES,FIG,GFIG,TAB,SUP,RDA)) dir.create(d, recursive=TRUE, showWarnings=FALSE)

LAB <- c(GSM359332=0,GSM359753=0,GSM359754=0,GSM359755=0,GSM359758=1,GSM359759=1,GSM359760=1,
         GSM305784=0,GSM305785=0,GSM305786=1,GSM305787=1)
COMMON <- c("GULP1","GATAD2A","ACSS1","LOC100506530","CNTN4","LINC00485",
            "WNT6","LOC101928461","IL20RA","VWC2","RNF128","TTC6")
HUB  <- c("WNT6","FZD9","DKK4","FZD6","FZD3")
FEAT <- unique(c(COMMON,HUB))
NBOOT <- 1000; NCLOUD <- 500; NBOOT_CI <- 5000; SEED <- 1
COLG <- c(Control="#2E86C1", Anthrax="#C0392B")
BASE <- 13

## ============================ READERS ============================
read_filtered <- function(path){
  if (!file.exists(path)) return(NULL)
  df <- as.data.frame(fread(path))
  sym <- grep("^Gene[._ ]?symbol$|^Symbol$", colnames(df), ignore.case=TRUE, value=TRUE)[1]
  gsm <- grep("^GSM[0-9]+", colnames(df), value=TRUE)
  if (is.na(sym) || !length(gsm)) return(NULL)
  d <- df[,c(sym,gsm)]; s <- trimws(as.character(d[[sym]]))
  k <- s!="" & !is.na(s) & s!="NA" & !grepl("///",s,fixed=TRUE)
  d <- d[k,]; s <- s[k]; d[gsm] <- lapply(d[gsm], as.numeric)
  sm <- rowsum(as.matrix(d[gsm]), s); as.data.frame(sm/as.vector(table(s)[rownames(sm)])) }
read_plain <- function(path){
  if (!file.exists(path)) return(NULL)
  dt <- fread(path); rn <- dt[[1]]; dt[[1]] <- NULL
  m <- as.data.frame(dt); rownames(m) <- make.unique(as.character(rn))
  m[] <- lapply(m, function(v) suppressWarnings(as.numeric(v))); m }
fix_scale <- function(M,nm){ if (min(M,na.rm=TRUE)>=0 && max(M,na.rm=TRUE)>50){
    cat("    ",nm,": log2(x+1)\\n"); return(log2(M+1)) }
  cat("    ",nm,": no transform\\n"); M }

PIPE <- list(
  P1=list(tag="P1_filtered_data",
    A=function() read_filtered(paste0(EV,"Dis_GSE12131/filtered_data_GSE12131.csv")),
    B=function() read_filtered(paste0(EV,"Dis_GSE14390/filtered_data_GSE14390.csv"))),
  P2=list(tag="P2_datasets_csv",
    A=function() read_plain(paste0(DATA,"GSE12131_gene_expression.csv")),
    B=function() read_plain(paste0(DATA,"GSE14390_gene_expression.csv"))))

## ======================= PART 1: PANEL BOOTSTRAP =======================
run_pipe <- function(p){
  A <- p$A(); B <- p$B(); if (is.null(A)||is.null(B)){cat("  [",p$tag,"] missing\\n"); return(NULL)}
  cat("  [",p$tag,"]\\n"); A <- fix_scale(A,"GSE12131"); B <- fix_scale(B,"GSE14390")
  feat <- intersect(FEAT, intersect(rownames(A),rownames(B)))
  Z <- t(scale(t(cbind(A[feat,,drop=FALSE], B[feat,,drop=FALSE])))); Z[is.na(Z)] <- 0
  org <- c(rep("GSE12131",ncol(A)),rep("GSE14390",ncol(B))); names(org) <- colnames(Z)
  grp <- ifelse(LAB[colnames(Z)]==1,"Anthrax","Control")
  cat(sprintf("     -> %d features | %d samples (%d/%d)\\n",length(feat),ncol(Z),
              sum(grp=="Anthrax"),sum(grp=="Control")))
  set.seed(SEED)
  pv <- pvclust(Z,method.hclust="average",method.dist="correlation",
                nboot=NBOOT,parallel=FALSE,quiet=TRUE)
  e <- pv$edges
  tb <- data.frame(Pipeline=p$tag,Cluster_edge=seq_len(nrow(e)),AU=round(e$au,4),
    BP=round(e$bp,4),SE_AU=round(e$se.au,4),P_value=signif(1-e$au,3),
    Supported_AU95=ifelse(e$au>=0.95,"Yes","No"),Is_root=c(rep("No",nrow(e)-1),"Yes"))
  list(pv=pv,Z=Z,origin=org,group=grp,tab=tb,feat=feat,A=A,B=B) }

cat("Running panel bootstrap (two pipelines)...\\n")
R <- lapply(PIPE, run_pipe); R <- R[!sapply(R,is.null)]; stopifnot(length(R)>=1)
allTab <- do.call(rbind, lapply(R, function(x) x$tab))
write.csv(allTab, paste0(TAB,"Table_bootstrap_confidence.csv"), row.names=FALSE)
cat("\\n--- bootstrap confidence ---\\n"); print(allTab, row.names=FALSE)

if (length(R)==2){
  a <- R[[1]]$tab; b <- R[[2]]$tab; n <- min(nrow(a),nrow(b))
  sens <- data.frame(Cluster_edge=1:n,AU_P1=a$AU[1:n],AU_P2=b$AU[1:n],
    Delta_AU=round(abs(a$AU[1:n]-b$AU[1:n]),4),
    Support_P1=a$Supported_AU95[1:n],Support_P2=b$Supported_AU95[1:n])
  sens$Agreement <- ifelse(sens$Support_P1==sens$Support_P2,"consistent","FLIPPED")
  write.csv(sens, paste0(SUP,"TableS_preprocessing_sensitivity.csv"), row.names=FALSE)
  cat("\\n--- preprocessing sensitivity ---\\n"); print(sens, row.names=FALSE)
  cat(sprintf("\\n>>> edges flipping: %d of %d | max |dAU| = %.3f\\n",
      sum(sens$Agreement=="FLIPPED"),n,max(sens$Delta_AU))) }

m <- R[[1]]; pv <- m$pv; Z <- m$Z; origin <- m$origin; group <- m$group
k2 <- cutree(pv$hclust,k=2)
cmp <- data.frame(Sample=names(k2),Cluster=as.integer(k2),
                  Dataset=origin[names(k2)],Group=group[names(k2)])
agr <- function(t) max(sum(diag(t)),sum(t)-sum(diag(t)))/sum(t)
aD <- agr(table(cmp$Cluster,cmp$Dataset)); aG <- agr(table(cmp$Cluster,cmp$Group))
DRIVER <- if (aD>aG) "BATCH (dataset)" else if (aG>aD) "BIOLOGY (group)" else "AMBIGUOUS"
write.csv(cmp, paste0(SUP,"TableS_cluster_membership.csv"), row.names=FALSE)
cat("\\n--- 2-cluster solution ---\\n"); print(cmp,row.names=FALSE)
cat("\\nCluster x GROUP:\\n"); print(table(cmp$Cluster,cmp$Group))
cat(sprintf("\\n>>> DATASET %.2f | GROUP %.2f -> %s\\n",aD,aG,DRIVER))

## ---------- PANEL A: MDS with ggrepel (no overlapping labels) ----------
set.seed(SEED)
cloud <- do.call(rbind, lapply(seq_len(NCLOUD), function(b){
  idx <- sample(nrow(Z),nrow(Z),replace=TRUE)
  d <- dist(t(Z[idx,,drop=FALSE])); if (any(!is.finite(d))) return(NULL)
  mds <- cmdscale(d,k=2)
  data.frame(Coord1=mds[,1],Coord2=mds[,2],Sample=rownames(mds),
             Dataset=origin[rownames(mds)],Group=group[rownames(mds)]) }))
cen <- aggregate(cbind(Coord1,Coord2) ~ Sample+Group+Dataset, cloud, mean)

pA <- ggplot(cloud, aes(Coord1,Coord2))+
  geom_point(aes(colour=Group,shape=Dataset), size=1.0, alpha=.14, show.legend=FALSE)+
  stat_ellipse(aes(group=Sample), type="norm", level=.68,
               linewidth=.35, colour="grey45", alpha=.85)+
  geom_point(data=cen, aes(fill=Group,shape=Dataset), size=4.6, stroke=.9, colour="black")+
  geom_text_repel(data=cen, aes(label=Sample, colour=Group),
                  size=3.5, fontface="bold", show.legend=FALSE,
                  box.padding=.75, point.padding=.45, min.segment.length=0,
                  segment.size=.35, segment.colour="grey40",
                  max.overlaps=Inf, seed=SEED, force=14)+
  scale_colour_manual(values=COLG)+ scale_fill_manual(values=COLG)+
  scale_shape_manual(values=c(GSE12131=21, GSE14390=24))+
  coord_cartesian(clip="off")+
  theme_classic(base_size=BASE)+
  theme(legend.position="bottom", legend.box="horizontal", legend.title=element_blank(),
        axis.text=element_text(colour="black"), axis.title=element_text(face="bold"),
        plot.subtitle=element_text(size=BASE-2.5, colour="grey30"),
        plot.margin=margin(12,20,8,10))+
  guides(fill=guide_legend(override.aes=list(shape=21,size=4.2)),
         shape=guide_legend(override.aes=list(size=4.2, fill="grey70")))+
  labs(x="Coordinate 1", y="Coordinate 2",
       subtitle=paste0(NCLOUD," gene-resampling bootstrap replicates per sample (68% ellipses)"))
ggsave(paste0(FIG,"Fig_Bootstrap_PanelA_MDS.png"), pA, width=7.2,height=6.4,dpi=600,bg="white")
ggsave(paste0(FIG,"Fig_Bootstrap_PanelA_MDS.pdf"), pA, width=7.2,height=6.4,bg="white")

## ---------- PANEL B: ggplot dendrogram (AU in boxes, nothing overlaps) ----------
hc  <- pv$hclust
dd  <- dendro_data(as.dendrogram(hc), type="rectangle")
seg <- segment(dd); lf <- label(dd)
lf$Group <- group[as.character(lf$label)]
## node coordinates for AU labels
A <- hc$merge; nn <- nrow(A); zz <- match(seq_len(nn+1), hc$order); xax <- numeric(nn)
for (i in seq_len(nn)){
  v <- c(if (A[i,1]<0) zz[-A[i,1]] else xax[A[i,1]],
         if (A[i,2]<0) zz[-A[i,2]] else xax[A[i,2]])
  xax[i] <- mean(v) }
nd <- data.frame(x=xax, y=hc$height, AU=round(pv$edges$au*100),
                 Sup=ifelse(pv$edges$au>=0.95,"AU \u2265 95","AU < 95"))
nd <- nd[nd$y < max(nd$y)*0.999 | seq_len(nrow(nd))==nrow(nd), ]   # keep all incl. root
ymax <- max(seg$y)
pB <- ggplot()+
  geom_segment(data=seg, aes(x=x,y=y,xend=xend,yend=yend), linewidth=.55, colour="grey25")+
  geom_label(data=nd, aes(x=x,y=y,label=AU,colour=Sup), fill="white",
             label.size=.28, label.padding=unit(.14,"lines"),
             size=3.5, fontface="bold", show.legend=TRUE)+
  geom_tile(data=lf, aes(x=x, y=-ymax*0.055, fill=Group), width=.62, height=ymax*0.05,
            colour="black", linewidth=.25)+
  geom_text(data=lf, aes(x=x, y=-ymax*0.105, label=label), angle=90, hjust=1,
            size=3.3, colour="black")+
  scale_colour_manual(values=c(`AU \u2265 95`="#C0392B", `AU < 95`="grey35"), name=NULL)+
  scale_fill_manual(values=COLG, name=NULL)+
  scale_y_continuous(name="Height", limits=c(-ymax*0.52, ymax*1.12),
                     breaks=pretty(c(0,ymax)))+
  coord_cartesian(clip="off")+
  theme_classic(base_size=BASE)+
  theme(axis.title.x=element_blank(), axis.text.x=element_blank(),
        axis.ticks.x=element_blank(), axis.line.x=element_blank(),
        axis.text.y=element_text(colour="black"), axis.title.y=element_text(face="bold"),
        legend.position="bottom", legend.box="horizontal",
        plot.title=element_text(face="bold", hjust=.5, size=BASE+1),
        plot.subtitle=element_text(hjust=.5, size=BASE-2.5, colour="grey30"),
        plot.margin=margin(12,14,8,12))+
  guides(colour=guide_legend(order=2, override.aes=list(label="95", fill="white")),
         fill=guide_legend(order=1, override.aes=list(colour="black")))+
  labs(title="Bootstrap sample dendrogram (AU %)",
       subtitle=sprintf("correlation distance, average linkage, %d replicates  |  split driven by %s",
                        NBOOT, DRIVER))
ggsave(paste0(FIG,"Fig_Bootstrap_PanelB_dendrogram.png"), pB, width=8.6,height=6.6,dpi=600,bg="white")
ggsave(paste0(FIG,"Fig_Bootstrap_PanelB_dendrogram.pdf"), pB, width=8.6,height=6.6,bg="white")

## ---------- combined A|B (same ggplot objects, no raster) ----------
comb <- (pA | pB) + plot_layout(widths=c(1,1.12)) +
  plot_annotation(tag_levels="A") &
  theme(plot.tag=element_text(face="bold", size=18))
ggsave(paste0(FIG,"Fig_Bootstrap_combined.png"), comb, width=15.5,height=6.8,dpi=600,bg="white")
ggsave(paste0(FIG,"Fig_Bootstrap_combined.pdf"), comb, width=15.5,height=6.8,bg="white")

## ---------- hub-only dendrogram (guarded, same clean style) ----------
ph <- intersect(HUB, rownames(Z)); hub_ok <- FALSE
if (length(ph)>=3){
  set.seed(SEED+1)
  pvh <- tryCatch(suppressWarnings(pvclust(Z[ph,,drop=FALSE],method.hclust="average",
         method.dist="correlation",nboot=NBOOT,parallel=FALSE,quiet=TRUE)),error=function(e) NULL)
  if (!is.null(pvh) && all(is.finite(pvh$edges$au)) && sd(pvh$edges$au)>1e-6){
    hub_ok <- TRUE
    hc2 <- pvh$hclust; dd2 <- dendro_data(as.dendrogram(hc2),type="rectangle")
    s2 <- segment(dd2); l2 <- label(dd2); l2$Group <- group[as.character(l2$label)]
    A2 <- hc2$merge; n2 <- nrow(A2); z2 <- match(seq_len(n2+1),hc2$order); x2 <- numeric(n2)
    for (i in seq_len(n2)){ v <- c(if (A2[i,1]<0) z2[-A2[i,1]] else x2[A2[i,1]],
                                   if (A2[i,2]<0) z2[-A2[i,2]] else x2[A2[i,2]]); x2[i] <- mean(v) }
    nd2 <- data.frame(x=x2,y=hc2$height,AU=round(pvh$edges$au*100),
                      Sup=ifelse(pvh$edges$au>=0.95,"AU \u2265 95","AU < 95"))
    ym2 <- max(s2$y)
    pH <- ggplot()+
      geom_segment(data=s2,aes(x=x,y=y,xend=xend,yend=yend),linewidth=.55,colour="grey25")+
      geom_label(data=nd2,aes(x=x,y=y,label=AU,colour=Sup),fill="white",label.size=.28,
                 size=3.4,fontface="bold")+
      geom_tile(data=l2,aes(x=x,y=-ym2*0.055,fill=Group),width=.62,height=ym2*0.05,
                colour="black",linewidth=.25)+
      geom_text(data=l2,aes(x=x,y=-ym2*0.105,label=label),angle=90,hjust=1,size=3.2)+
      scale_colour_manual(values=c(`AU \u2265 95`="#C0392B",`AU < 95`="grey35"),name=NULL)+
      scale_fill_manual(values=COLG,name=NULL)+
      scale_y_continuous(name="Height",limits=c(-ym2*0.52,ym2*1.12))+
      coord_cartesian(clip="off")+ theme_classic(base_size=BASE)+
      theme(axis.title.x=element_blank(),axis.text.x=element_blank(),
            axis.ticks.x=element_blank(),axis.line.x=element_blank(),
            legend.position="bottom",plot.title=element_text(face="bold",hjust=.5))+
      labs(title="Bootstrap dendrogram - top-5 hub genes only")
    ggsave(paste0(SUP,"FigS_bootstrap_hub_only.png"),pH,width=8.6,height=6.6,dpi=600,bg="white")
    write.csv(data.frame(Cluster_edge=seq_len(nrow(pvh$edges)),AU=round(pvh$edges$au,4),
      BP=round(pvh$edges$bp,4)), paste0(SUP,"TableS_bootstrap_hub_only.csv"), row.names=FALSE)
  } else cat("\\n>>> hub-only bootstrap DEGENERATE - not reported\\n") }

## ================= PART 2: PER-HUB-GENE BOOTSTRAP =================
cat("\\n================ PER-HUB-GENE BOOTSTRAP ================\\n")
EA <- m$A; EB <- m$B
zrow <- function(M) t(apply(M,1,function(v){ s <- sd(v,na.rm=TRUE)
  if (!is.finite(s)||s==0) rep(0,length(v)) else (v-mean(v,na.rm=TRUE))/s }))
gg <- intersect(HUB, intersect(rownames(EA),rownames(EB)))
cat("hub genes:",length(gg),"-",paste(gg,collapse=", "),"\\n")
ZG <- cbind(zrow(EA[gg,,drop=FALSE]), zrow(EB[gg,,drop=FALSE]))
colnames(ZG) <- c(colnames(EA),colnames(EB)); rownames(ZG) <- gg
grpG <- ifelse(LAB[colnames(ZG)]==1,"Anthrax","Control")
dstG <- c(rep("GSE12131",ncol(EA)),rep("GSE14390",ncol(EB)))

boot_ci <- function(x,seed){ x <- x[is.finite(x)]; n <- length(x)
  if (n<2) return(c(mean=ifelse(n==1,x,NA),lo=NA,hi=NA,n=n))
  set.seed(seed); bm <- replicate(NBOOT_CI, mean(sample(x,n,replace=TRUE)))
  c(mean=mean(x),lo=unname(quantile(bm,.025)),hi=unname(quantile(bm,.975)),n=n) }

thm <- theme_bw(BASE)+theme(panel.grid.minor=element_blank(),
  panel.grid.major.x=element_blank(), axis.text=element_text(colour="black"),
  axis.title=element_text(face="bold"), plot.title=element_text(face="bold",hjust=.5,size=BASE))

hubRows <- list(); hubPlots <- list()
for (x in gg){
  d <- data.frame(Sample=colnames(ZG),Group=factor(grpG,levels=c("Control","Anthrax")),
                  Dataset=dstG,Expr=as.numeric(ZG[x,]))
  d <- d[is.finite(d$Expr),]
  cC <- boot_ci(d$Expr[d$Group=="Control"],1); cA <- boot_ci(d$Expr[d$Group=="Anthrax"],2)
  ci <- data.frame(Group=factor(c("Control","Anthrax"),levels=c("Control","Anthrax")),
    Mean=c(cC["mean"],cA["mean"]),lo=c(cC["lo"],cA["lo"]),hi=c(cC["hi"],cA["hi"]),
    N=c(cC["n"],cA["n"]))
  ovl <- !(ci$hi[1] < ci$lo[2] || ci$hi[2] < ci$lo[1])
  wt  <- suppressWarnings(wilcox.test(Expr ~ Group, data=d)$p.value)
  rng <- range(c(d$Expr, ci$lo, ci$hi, 0), na.rm=TRUE); pad <- diff(rng)*0.18

  p1 <- ggplot(d, aes(Group,Expr))+
    geom_hline(yintercept=0, colour="grey80", linewidth=.4)+
    geom_boxplot(aes(fill=Group), alpha=.55, outlier.shape=NA, width=.56, linewidth=.55)+
    geom_jitter(aes(fill=Group,shape=Dataset), width=.12, height=0, size=3, alpha=.95,
                colour="black", stroke=.45)+
    scale_fill_manual(values=COLG, guide="none")+
    scale_shape_manual(values=c(GSE12131=21,GSE14390=24), name=NULL)+
    scale_y_continuous(limits=c(rng[1]-pad, rng[2]+pad))+
    thm+theme(legend.position="bottom")+
    labs(title="A. Expression distribution", x=NULL, y="Expression (z-score)")
  p2 <- ggplot(ci, aes(Group,Mean,fill=Group))+
    geom_hline(yintercept=0, colour="grey80", linewidth=.4)+
    geom_col(width=.56, colour="black", alpha=.85, linewidth=.55)+
    geom_errorbar(aes(ymin=lo,ymax=hi), width=.17, linewidth=.8, na.rm=TRUE)+
    geom_text(aes(label=paste0("n = ",N), y=ifelse(Mean>=0, hi, lo)),
              vjust=ifelse(ci$Mean>=0,-0.9,1.7), size=3.5, fontface="italic")+
    scale_fill_manual(values=COLG, guide="none")+
    scale_y_continuous(limits=c(rng[1]-pad, rng[2]+pad))+
    thm+labs(title=paste0("B. Bootstrap 95% CI",
                          if (!ovl) "  (non-overlapping)" else ""),
             x=NULL, y="Mean z-score")
  pp <- (p1|p2)+plot_annotation(title=x,
    subtitle=sprintf("CI overlap: %s   |   Wilcoxon p = %.3f   |   %s",
      ifelse(ovl,"yes","NO"), wt,
      ifelse(ci$Mean[2]>ci$Mean[1],"higher in anthrax","lower in anthrax")),
    theme=theme(plot.title=element_text(face="bold.italic",size=16,hjust=.5),
                plot.subtitle=element_text(size=10,hjust=.5,colour="grey25")))
  hubPlots[[x]] <- pp
  ggsave(paste0(GFIG,x,"_bootstrap.png"),pp,width=9,height=5,dpi=600,bg="white")
  ggsave(paste0(GFIG,x,"_bootstrap.pdf"),pp,width=9,height=5,bg="white")
  cat(sprintf("  %-6s ctrl %6.3f [%6.3f,%6.3f] | anth %6.3f [%6.3f,%6.3f] | overlap %-3s | p=%.3f\\n",
      x,ci$Mean[1],ci$lo[1],ci$hi[1],ci$Mean[2],ci$lo[2],ci$hi[2],ifelse(ovl,"yes","NO"),wt))
  hubRows[[x]] <- data.frame(Gene=x,Group=ci$Group,N=ci$N,Mean_z=round(ci$Mean,4),
    CI_low=round(ci$lo,4),CI_high=round(ci$hi,4),CI_overlap=ifelse(ovl,"Yes","No"),
    Wilcoxon_p=round(wt,4),
    Direction=ifelse(ci$Mean[2]>ci$Mean[1],"Higher in anthrax","Lower in anthrax")) }

hubS <- do.call(rbind,hubRows); rownames(hubS) <- NULL
write.csv(hubS, paste0(TAB,"Table_hubgene_bootstrapCI_by_group.csv"), row.names=FALSE)
cat("\\n--- hub gene summary ---\\n"); print(hubS,row.names=FALSE)
sepg <- unique(hubS$Gene[hubS$CI_overlap=="No"])
cat("\\n>>> non-overlapping CI:", if (length(sepg)) paste(sepg,collapse=", ") else "none","\\n")

ovf <- wrap_plots(hubPlots,ncol=1)+plot_annotation(
  title="Hub gene bootstrap analysis (Anthrax vs Control, z-scored within dataset)",
  theme=theme(plot.title=element_text(face="bold",size=18,hjust=.5)))
ggsave(paste0(GFIG,"AllHubGenes_overview.png"),ovf,width=9.5,height=5*length(hubPlots),
       dpi=400,bg="white",limitsize=FALSE)
ggsave(paste0(GFIG,"AllHubGenes_overview.pdf"),ovf,width=9.5,height=5*length(hubPlots),
       bg="white",limitsize=FALSE)

## ============================ VALIDITY REPORT ============================
nonroot <- subset(m$tab, Is_root=="No")
rep <- data.frame(
  Item=c("Datasets","Samples (total)","Anthrax / Control","Panel genes used",
    "Bootstrap replicates (panel)","Bootstrap replicates (per-gene CI)",
    "Internal nodes (excl. root)","Nodes AU >= 0.95","Lowest AU","Pipelines compared",
    "Edges flipping support","Cluster agreement: dataset","Cluster agreement: group",
    "2-cluster split driven by","Hub-only dendrogram usable",
    "Hub genes with non-overlapping CI","Overall interpretation"),
  Value=c("GSE12131 + GSE14390",ncol(Z),
    paste0(sum(group=="Anthrax")," / ",sum(group=="Control")),
    paste0(length(m$feat)," of ",length(FEAT)),NBOOT,NBOOT_CI,
    nrow(nonroot),sum(nonroot$Supported_AU95=="Yes"),round(min(nonroot$AU),3),length(R),
    if (exists("sens")) sum(sens$Agreement=="FLIPPED") else NA,
    round(aD,2),round(aG,2),DRIVER,ifelse(hub_ok,"Yes","No (degenerate)"),
    ifelse(length(sepg),paste(sepg,collapse=", "),"none"),
    ifelse(DRIVER=="BIOLOGY (group)" &&
           sum(nonroot$Supported_AU95=="Yes")>=ceiling(nrow(nonroot)/2),
           "Usable as clustering-stability support",
           "Report as stability check only - NOT validation")))
write.csv(rep, paste0(SUP,"TableS_bootstrap_validity_report.csv"), row.names=FALSE)
cat("\\n================ VALIDITY REPORT ================\\n"); print(rep,right=FALSE)
save(R,pv,Z,origin,group,cmp,aD,aG,DRIVER,hubS,ZG,rep,file=paste0(RDA,"Bootstrap_Final.RData"))
cat("\\nSaved -> ",RES,"\\n")
''')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


(as ‘lib’ is unspecified)







	‘/tmp/Rtmppub1Jx/downloaded_packages’

(as ‘lib’ is unspecified)







	‘/tmp/Rtmppub1Jx/downloaded_packages’



Running panel bootstrap (two pipelines)...
  [ P1_filtered_data ]
     GSE12131 : log2(x+1)
     GSE14390 : no transform
     -> 16 features | 11 samples (5/6)
  [ P2_datasets_csv ]
     GSE12131 : log2(x+1)
     GSE14390 : no transform
     -> 16 features | 11 samples (5/6)

--- bootstrap confidence ---
         Pipeline Cluster_edge     AU     BP  SE_AU P_value Supported_AU95
 P1_filtered_data            1 0.7315 0.6638 0.0258  0.2690             No
 P1_filtered_data            2 0.9576 0.7864 0.0075  0.0424            Yes
 P1_filtered_data            3 0.9623 0.5636 0.0064  0.0377            Yes
 P1_filtered_data            4 0.9191 0.6925 0.0119  0.0809             No
 P1_filtered_data            5 0.8559 0.5428 0.0175  0.1440             No
 P1_filtered_data            6 0.7979 0.4753 0.0217  0.2020             No
 P1_filtered_data            7 0.7683 0.4745 0.0234  0.2320             No
 P1_filtered_data            8 0.9117 0.5751 0.0124  0.0883             No
 P1_filtered_data  


ℹ Please use the `linewidth` argument instead.
This warning is displayed once per session.
Call `lifecycle::last_lifecycle_warnings()` to see where this warning was
generated. 

 

 

